In [0]:
# ==============================================================================
# UNIVERSIDAD LATINOAMERICANA DE CIENCIA Y TECNOLOGÍA (ULACIT)
# Curso: Big Data y Tecnologías de Procesamiento
# Scripts de Ingestión y Almacenamiento (Capa Bronce y Plata - Medallion)
# ==============================================================================

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, trim, regexp_replace
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.functions import col, to_date, regexp_replace


In [0]:
# 0. Inicialización 
PATH_ORIGEN_CSV = "/Volumes/workspace/bigdata/test/retail_store_sales.csv"

print("=== CSV cargado correctamente ===")


=== CSV cargado correctamente ===


In [0]:
# ==============================================================================
# 1. CAPA DE INGESTA Y ALMACENAMIENTO BRONCE (Raw Data)
# Objetivo: Cargar los datos crudos
# ==============================================================================

# Definición del esquema original para asegurar una correcta lectura inicial en modo Batch
esquema_bronce = StructType([
    StructField("Transaction ID", StringType(), True),
    StructField("Customer ID", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Item", StringType(), True),
    StructField("Price Per Unit", DoubleType(), True),
    StructField("Quantity", DoubleType(), True),
    StructField("Total Spent", DoubleType(), True),
    StructField("Payment Method", StringType(), True),
    StructField("Location", StringType(), True),
    StructField("Transaction Date", StringType(), True),
    StructField("Discount Applied", StringType(), True)
])

print("\n[INFO] Leyendo archivo CSV estático en modo Batch...")
df_raw = spark.read.format("csv").option("header", "true").schema(esquema_bronce).load(PATH_ORIGEN_CSV)


[INFO] Leyendo archivo CSV estático en modo Batch...


In [0]:
df_storesales = spark.read.csv(
    PATH_ORIGEN_CSV,
    header=True,
    schema=esquema_bronce,   # <-- en vez de inferSchema=True
    encoding="UTF-8"         # <-- explicito: el dataset tiene tildes (San Jose, Limon)
)

print(f"BigMart CR Sales cargado: {df_storesales.count():,} filas, {len(df_storesales.columns)} columnas")
df_storesales.printSchema()
display(df_storesales.limit(10))

BigMart CR Sales cargado: 12,575 filas, 11 columnas
root
 |-- Transaction ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Price Per Unit: double (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- Total Spent: double (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Transaction Date: string (nullable = true)
 |-- Discount Applied: string (nullable = true)



Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,null
TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False
TXN_7482416,CUST_09,Patisserie,null,null,10.0,200.0,Credit Card,Online,2023-11-30,null
TXN_3652209,CUST_07,Food,Item_1_FOOD,5.0,8.0,40.0,Credit Card,In-store,2023-06-10,True
TXN_1372952,CUST_21,Furniture,null,33.5,null,null,Digital Wallet,In-store,2024-04-02,True
TXN_9728486,CUST_23,Furniture,Item_16_FUR,27.5,1.0,27.5,Credit Card,In-store,2023-04-26,False
TXN_2722661,CUST_25,Butchers,Item_22_BUT,36.5,3.0,109.5,Cash,Online,2024-03-14,False


In [0]:
#Mostrar el tipo de datos por columna
print(df_raw.dtypes)

[('Transaction ID', 'string'), ('Customer ID', 'string'), ('Category', 'string'), ('Item', 'string'), ('Price Per Unit', 'double'), ('Quantity', 'double'), ('Total Spent', 'double'), ('Payment Method', 'string'), ('Location', 'string'), ('Transaction Date', 'string'), ('Discount Applied', 'string')]


In [0]:
# Renombrar columnas para eliminar espacios (requerido por Delta Lake)
df_raw = df_raw.toDF(*[col_name.replace(" ", "") for col_name in df_raw.columns])

In [0]:
# Guardar en Capa Bronce usando formato Delta (estándar del enfoque Lakehouse)
print("[BRONCE] Guardando datos crudos en la Capa Bronce (Delta Lake)...")
df_raw.write.format("delta").mode("overwrite").saveAsTable("default.retail_sales_bronce")

print("[BRONCE] Tabla 'default.retail_sales_bronce' creada exitosamente.")

[BRONCE] Guardando datos crudos en la Capa Bronce (Delta Lake)...
[BRONCE] Tabla 'default.retail_sales_bronce' creada exitosamente.


In [0]:
from pyspark.sql.functions import to_date

# Convertir la fecha con la fecha de transacción a formato fecha y el porcentaje de descuento a porcentaje
df_datos_limpios = df_raw.withColumn(
    "TransactionDate",
    to_date(col("TransactionDate"), "yyyy-MM-dd")  # Cambia el formato de ambas columnas
)
# Verificar el esquema
df_datos_limpios.printSchema()

root
 |-- TransactionID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- PricePerUnit: double (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- TotalSpent: double (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- TransactionDate: date (nullable = true)
 |-- DiscountApplied: string (nullable = true)



In [0]:
# Estadisticas numericas con describe()
print("BIGMART CR SALES — ESTADISTICAS NUMERICAS")
df_raw.select('Item', 'Location', 'Quantity', 'Category', 'CustomerID').describe().show()

BIGMART CR SALES — ESTADISTICAS NUMERICAS
+-------+-----------+--------+------------------+----------+----------+
|summary|       Item|Location|          Quantity|  Category|CustomerID|
+-------+-----------+--------+------------------+----------+----------+
|  count|      11362|   12575|             11971|     12575|     12575|
|   mean|       NULL|    NULL| 5.536379583994654|      NULL|      NULL|
| stddev|       NULL|    NULL|2.8578828340802995|      NULL|      NULL|
|    min|Item_10_BEV|In-store|               1.0| Beverages|   CUST_01|
|    max| Item_9_PAT|  Online|              10.0|Patisserie|   CUST_25|
+-------+-----------+--------+------------------+----------+----------+



In [0]:
#Contar coantidad de nulos por columna

from pyspark.sql.functions import count

total_rows = df_raw.count()

null_counts = df_raw.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_raw.columns
]).collect()[0].asDict()

print(f"\n{'Columna':<25}{'Nulos':<10}{'%':<8}")
for col_name, null_count in null_counts.items():
    pct = 100 * null_count / total_rows
    flag = "  <-- atencion" if pct > 0 else ""
    print(f"{col_name:<25}{null_count:<10}{pct:>5.2f}%{flag}")


Columna                  Nulos     %       
TransactionID            0          0.00%
CustomerID               0          0.00%
Category                 0          0.00%
Item                     1213       9.65%  <-- atencion
PricePerUnit             609        4.84%  <-- atencion
Quantity                 604        4.80%  <-- atencion
TotalSpent               604        4.80%  <-- atencion
PaymentMethod            0          0.00%
Location                 0          0.00%
TransactionDate          0          0.00%
DiscountApplied          4199      33.39%  <-- atencion


*Podemos ver que hay 33.39% de nulos en la columna DiscountApplied, por lo que podemos limpiarla.*
**Al igual que que en las columnas Item, PricePerUnit, Quantity y TotalSpent**
# Limpiar datos en null


In [0]:
from pyspark.sql.functions import avg, col, when, count, desc

# Obtener la moda para columnas string
moda_payment = df_raw.groupBy("PaymentMethod").count().orderBy(desc("count")).first()["PaymentMethod"]
moda_item = df_raw.groupBy("Item").count().orderBy(desc("count")).first()["Item"]
moda_discount = df_raw.groupBy("DiscountApplied").count().orderBy(desc("count")).first()["DiscountApplied"]

# Obtener el promedio para Quantity (double)
promedio_quantity = df_raw.select(avg(col("Quantity"))).first()[0]
promedio_PricePerUnit = df_raw.select(avg(col("PricePerUnit"))).first()[0]
promedio_TotalSpent = df_raw.select(avg(col("TotalSpent"))).first()[0]
# Reemplazar los valores nulos por la moda o promedio según corresponda
df_raw = df_raw.withColumn(
    "PaymentMethod",
    when(col("PaymentMethod").isNull(), moda_payment).otherwise(col("PaymentMethod"))
).withColumn(
    "Item",
    when(col("Item").isNull(), moda_item).otherwise(col("Item"))
).withColumn(
    "DiscountApplied",
    when(col("DiscountApplied").isNull(), moda_discount).otherwise(col("DiscountApplied"))
).withColumn(
    "Quantity",
    when(col("Quantity").isNull(), promedio_quantity).otherwise(col("Quantity"))
).withColumn(
    "PricePerUnit",
    when(col("PricePerUnit").isNull(), promedio_PricePerUnit).otherwise(col("PricePerUnit"))
).withColumn(
    "TotalSpent",
    when(col("TotalSpent").isNull(), promedio_TotalSpent).otherwise(col("TotalSpent"))
)

display(df_raw.limit(10))

TransactionID,CustomerID,Category,Item,PricePerUnit,Quantity,TotalSpent,PaymentMethod,Location,TransactionDate,DiscountApplied
TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,True
TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False
TXN_7482416,CUST_09,Patisserie,Unknown,23.365911749958215,10.0,200.0,Credit Card,Online,2023-11-30,True
TXN_3652209,CUST_07,Food,Item_1_FOOD,5.0,8.0,40.0,Credit Card,In-store,2023-06-10,True
TXN_1372952,CUST_21,Furniture,Unknown,33.5,5.536379583994654,129.6525770612313,Digital Wallet,In-store,2024-04-02,True
TXN_9728486,CUST_23,Furniture,Item_16_FUR,27.5,1.0,27.5,Credit Card,In-store,2023-04-26,False
TXN_2722661,CUST_25,Butchers,Item_22_BUT,36.5,3.0,109.5,Cash,Online,2024-03-14,False


In [0]:
# DEMO 1.3: Duplicados exactos en BigMart CR

dup_count = df_raw.count() - df_raw.dropDuplicates().count()
print(f"\nDuplicados exactos (todas las columnas iguales): {dup_count}")
print("Estrategia: DROP directo con dropDuplicates() -- son filas 100% redundantes.")


Duplicados exactos (todas las columnas iguales): 0
Estrategia: DROP directo con dropDuplicates() -- son filas 100% redundantes.


In [0]:
 # Distribucion y outliers en la columna Quanttity

percentiles = df_raw.approxQuantile('Quantity', [0.01, 0.25, 0.5, 0.75, 0.95], 0.01)

print(f"\nPercentil 1:   {percentiles[0]:>10.2f}")
print(f"Percentil 25:  {percentiles[1]:>10.2f}")
print(f"Mediana (50):  {percentiles[2]:>10.2f}")
print(f"Percentil 75:  {percentiles[3]:>10.2f}")
print(f"Percentil 95:  {percentiles[4]:>10.2f}")

outliers = df_raw.filter(col('Quantity') > percentiles[4]).count()
print(f"\nPosibles outliers (> percentil 95): {outliers} ({100*outliers/df_raw.count():.2f}%)")


Percentil 1:         1.00
Percentil 25:        3.00
Mediana (50):        5.54
Percentil 75:        8.00
Percentil 95:       10.00

Posibles outliers (> percentil 95): 0 (0.00%)


In [0]:
from pyspark.sql.functions import (
    col, count, when, trim, isnan,
    min, max, avg, stddev, countDistinct
)
from pyspark.sql.types import StringType, NumericType

print("="*80)
print("PRUEBAS PRELIMINARES Y DE CALIDAD DE LOS DATOS")
print("="*80)

# 1. Dimensiones del DataFrame
print("\n1. DIMENSIONES DEL DATAFRAME")
print(f"Número de filas: {df_raw.count()}")
print(f"Número de columnas: {len(df_raw.columns)}")

PRUEBAS PRELIMINARES Y DE CALIDAD DE LOS DATOS

1. DIMENSIONES DEL DATAFRAME
Número de filas: 12575
Número de columnas: 11


In [0]:
# Verificar la cantidad de valores únicos
print("\n8. VALORES ÚNICOS POR COLUMNA")

for c in df_raw.columns:
    print(f"{c}: {df_raw.select(countDistinct(c)).first()[0]}")


8. VALORES ÚNICOS POR COLUMNA
TransactionID: 12575
CustomerID: 25
Category: 8
Item: 201
PricePerUnit: 26
Quantity: 11
TotalSpent: 228
PaymentMethod: 3
Location: 2
TransactionDate: 1114
DiscountApplied: 2


In [0]:
# Comparacion final de metricas
# ANTES: usar df_storesales (datos originales sin limpiar, con nombres de columna con espacios)
nulls_before = sum(
    df_storesales.filter(col(c).isNull()).count() for c in ['Item', 'Payment Method']
)
# DESPUES: usar df_raw (datos limpios, con nombres de columna sin espacios)
nulls_after = sum(
    df_raw.filter(col(c).isNull()).count() for c in ['Item', 'PaymentMethod']
)
Quantity_before = df_storesales.select('Quantity').distinct().count()
Quantity_after = df_raw.select('Quantity').distinct().count()

metrics_df = pd.DataFrame([
    {
        'Dataset': 'store_sales',
        'Filas originales': df_storesales.count(),
        'Filas finales': df_raw.count(),
        'Nulls criticos (antes)': nulls_before,
        'Nulls criticos (despues)': nulls_after,
        'Quantity unicas (antes)': Quantity_before,
        'Quantity unicas (despues)': Quantity_after,
    }
])

print("METRICAS DE CALIDAD — ANTES VS DESPUES")
display(metrics_df)

METRICAS DE CALIDAD — ANTES VS DESPUES


Dataset,Filas originales,Filas finales,Nulls criticos (antes),Nulls criticos (despues),Quantity unicas (antes),Quantity unicas (despues)
store_sales,12575,12575,1213,0,11,11
